In [ ]:
### Grayscale, Blur, Threshold, Contour

In [ ]:
import marimo as mo

# Interactive Area Calc

## Run once (usually autorun) to grab 3 files and set up variables.

### Autorun stuff probably only happens once, also functions, etc.

### Get some files and display info and images

### Contour and visualize

In [ ]:
import tkinter as tk
from tkinter import filedialog
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt


# Function to upload file(s)
def upload_file():
    file_path = filedialog.askopenfilenames(
        title="Select File(s)",
        filetypes=[("Tif;Tiff", ("*.tif *.tiff")), ("All files", "*.*")],
    )
    return file_path

In [ ]:
uploaded = list(upload_file())

# Create subplots for displaying images in a row
figo, axeso = plt.subplots(
    1, len(uploaded), figsize=(fsx := 15, fsy := 5)
)  # Adjust as needed

for i, fno in enumerate(uploaded):
    # Load image
    imageo = cv2.imread(fno)
    # Print image details
    print(
        f"Image size: {imageo.size} (width, height) for {os.path.basename(fno)}"
    )
    print(f"Image shape: {imageo.shape}")  # Print shape NumPy array
    print(f"Image dtype: {imageo.dtype}")  # Print data type for NumPy array
    # If the image is in RGB mode, print the color details
    if len(imageo.shape) == 3 and imageo.shape[2] == 3:
        # Get the unique colors in the image
        unique_colorso = np.unique(
            np.array(imageo).reshape(-1, np.array(imageo).shape[2]), axis=0
        )
        print(f"Number of unique colors: {len(unique_colorso)}")
        # print(f"Sample colors: {unique_colors[:5]}")  # Print first 5

    # Display image in the corresponding subplot
    axeso[i].imshow(imageo)  # alternatively:
    # axes[i].imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB)) # Convert to RGB
    # axes[i].axis("off")  # Hide axes
    axeso[i].set_title(os.path.basename(fno))

# Show Original Images
figo.suptitle("Original Images", fontsize=14)
plt.tight_layout()  # Adjust subplot spacing
plt.show()

In [ ]:
grayscale_images = {}  # empty dictionary
figg, axesg = plt.subplots(1, len(uploaded), figsize=(fsx, fsy))

for j, fng in enumerate(uploaded):
    imageg = cv2.imread(fng)
    # Convert to grayscale
    gray = cv2.cvtColor(imageg, cv2.COLOR_BGR2GRAY)
    # Add grayscale image to the dictionary with filename as key
    grayscale_images[fng] = gray

    # # Print image details
    # print(
    #     f"Image size: {gray.size} (width, height) for {os.path.basename(fng)}"
    # )
    # print(f"Image shape: {gray.shape}")
    # print(f"Image dtype: {gray.dtype}")

    # # If the image is in RGB mode, print the color details
    # if len(gray.shape) == 3 and gray.shape[2] == 3:
    #     # Get the unique colors in the image
    #     unique_colorsg = np.unique(
    #         np.array(gray).reshape(-1, np.array(gray).shape[2]), axis=0
    #     )
    #     print(f"Number of unique colors: {len(unique_colorsg)}")

    axesg[j].imshow(gray, cmap="gray")  # , cmap='gray'
    axesg[j].set_title(os.path.basename(fng))

# Show grayscale Images
figg.suptitle("Grayscale Images", fontsize=14)
plt.tight_layout()  # Adjust subplot spacing
plt.show()

In [ ]:
blurred_images = {}  # empty dictionary

figb, axesb = plt.subplots(1, len(uploaded), figsize=(fsx, fsy))

for k, (fnb, grayb) in enumerate(grayscale_images.items()):
    # Apply Gaussian blur to the grayscale image
    blurred = cv2.GaussianBlur(grayb, (5, 5), 0)
    # add blurred image to dictionary
    blurred_images[fnb] = blurred

    axesb[k].imshow(blurred, cmap="gray")  # , cmap='gray'
    axesb[k].set_title(os.path.basename(fnb))

# Show blurred Images
figb.suptitle("Blurred Images", fontsize=14)
plt.tight_layout()  # Adjust subplot spacing
plt.show()

In [ ]:
threshed_images = {}  # empty dictionary

figt, axest = plt.subplots(1, len(uploaded), figsize=(fsx, fsy))

for l, (fnt, blurredt) in enumerate(blurred_images.items()):
    _, thresh = cv2.threshold(blurredt, 25, 255, cv2.THRESH_BINARY)
    # add threshold image to dictionary
    threshed_images[fnt] = thresh

    axest[l].imshow(thresh, cmap="gray")  # , cmap='gray'
    axest[l].set_title(os.path.basename(fnt))

# Show Threshold Images
figt.suptitle("Threshold Images", fontsize=14)
plt.tight_layout()  # Adjust subplot spacing
plt.show()

In [ ]:
contoured_images = {}  # empty dictionary

figc, axesc = plt.subplots(1, len(uploaded), figsize=(fsx, fsy))


for m, (fnc, threshc) in enumerate(threshed_images.items()):
    # Find contours
    contours, _ = cv2.findContours(
        threshc, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    # Add contoured image to dictionary
    contoured_images[fnc] = contours

    # Draw contours (optional, for visualization)
    contour = cv2.drawContours(
        cv2.cvtColor(threshc.copy(), cv2.COLOR_GRAY2BGR),
        contours,
        -1,
        (0, 255, 0),  # Color is Green
        2,
    )

    # Create a mask of the contours
    mask = np.zeros_like(threshc, dtype=np.uint8)
    cv2.drawContours(
        mask, contours, -1, 255, thickness=cv2.FILLED
    )  # Fill contours with white (255)

    # Invert the mask to highlight background regions
    inverted_mask = cv2.bitwise_not(mask)

    # Color the background as orange
    background_colored = cv2.cvtColor(inverted_mask, cv2.COLOR_GRAY2BGR)
    background_colored[
        np.where((background_colored == [255, 255, 255]).all(axis=2))
    ] = [0, 165, 255]  # Orange in BGR format

    # Combine the original image with the colored background
    final_image = cv2.addWeighted(background_colored, 1, contour, 1, 0)

    # Convert final image to RGB for visualization
    final_image_rgb = cv2.cvtColor(final_image, cv2.COLOR_BGR2RGB)

    # contour_rgb = cv2.cvtColor(contour, cv2.COLOR_BGR2RGB)

    # # Print image details
    # print(f"Image size: {contour.size} (width, height)")
    # print(
    #     f"Image shape: {contour.shape}"
    # )  # Print shape instead of mode for NumPy array
    # print(f"Image dtype: {contour.dtype}")  # Print data type for NumPy array

    # # If the image is in RGB mode, print the color details
    # if len(contour.shape) == 3 and contour.shape[2] == 3:
    #     # Get the unique colors in the image
    #     unique_colors = np.unique(
    #         np.array(contour).reshape(-1, np.array(contour).shape[2]), axis=0
    #     )
    #     print(f"Number of unique colors: {len(unique_colors)}")

    total_area = (
        contour.shape[0] * contour.shape[1]
    )  # Total area of the image
    for n, (fnn, contoursn) in enumerate(contoured_images.items()):
        # Sum of areas of all spots
        spot_area = sum(cv2.contourArea(c) for c in contoursn)
        # Calculate percentage
        percentage = (spot_area / total_area) * 100
        result = f"Spots cover {percentage:.1f}% of area"

    axesc[m].imshow(final_image_rgb)  # , cmap='gray'
    axesc[m].set_title(os.path.basename(fnn))
    axesc[m].set_xlabel(result)  # Set the x-axis label

# Show Contoured Images
figc.suptitle("Contoured Images", fontsize=14)
plt.tight_layout()  # Adjust subplot spacing
plt.show()

## Interactive Part (adjust blur kernal, threshold value)

In [ ]:
# Create two sliders
x = mo.ui.slider(
    0,
    50,
    1,
    25,
    label="Threshold Value (above this, turn to Max)",
    show_value=True,
    full_width=False,
)
y = mo.ui.slider(
    0,
    255,
    5,
    130,
    label="Max Value (white = 255)",
    show_value=True,
    full_width=False,
)
z1 = mo.hstack(
    [x, y], justify="start", align="stretch"
)  # Arrange sliders in a row)

# Create more sliders
x2 = mo.ui.slider(
    1,
    51,
    2,
    5,
    label="blur kernal size",
    show_value=True,
    full_width=False,
)
y1 = mo.ui.slider(
    0,
    50,
    1,
    18,
    label="x",
    show_value=True,
    full_width=False,
)
y2 = mo.ui.slider(
    0,
    20,
    1,
    10,
    label="y",
    show_value=True,
    full_width=False,
)
z2 = mo.hstack(
    [x2, mo.md("Size of fig: "), y1, y2], justify="start", align="stretch"
)

z = mo.vstack(
    [z2, z1], justify="start", align="stretch"
)  # Arrange sliders in a column

z

In [ ]:
figi, axesi = plt.subplots(2, len(uploaded), figsize=(y1.value, y2.value))

iblurred_images = {}  # empty dictionary
for ifnb, igrayb in grayscale_images.items():
    # Apply Gaussian blur to the grayscale image
    iblurred = cv2.GaussianBlur(igrayb, (x2.value, x2.value), 0)
    # add blurred image to dictionary
    iblurred_images[ifnb] = iblurred

# ======================================

ithreshed_images = {}  # empty dictionary
for o, (fni, blurredi) in enumerate(iblurred_images.items()):
    _, ithresh = cv2.threshold(
        blurredi, xval := x.value, yval := y.value, cv2.THRESH_BINARY
    )  # 25 200

    ithreshed_images[fni] = ithresh

    # Convert grayscale image to RGB
    ithresh_rgb = cv2.cvtColor(ithresh, cv2.COLOR_GRAY2RGB)

    axesi[0, o].imshow(ithresh_rgb)  # , cmap='gray'
    axesi[0, o].set_title(os.path.basename(fni))
    axesi[0, o].set_xlabel(
        f"blur=({x2.value},{x2.value}) xThresh={xval}  yThresh={yval}"
    )

# ======================================

icontoured_images = {}  # empty dictionary
for im, (ifnc, ithreshc) in enumerate(ithreshed_images.items()):
    # Find contours
    icontours, _ = cv2.findContours(
        ithreshc, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )
    # Add contoured image to dictionary
    icontoured_images[ifnc] = icontours
    # Draw contours (optional, for visualization)
    icontour = cv2.drawContours(
        cv2.cvtColor(ithreshc.copy(), cv2.COLOR_GRAY2BGR),
        icontours,
        -1,
        (0, 255, 0),  # Color is Green
        2,
    )

    # Create a mask of the contours
    imask = np.zeros_like(ithreshc, dtype=np.uint8)
    cv2.drawContours(
        imask, icontours, -1, 255, thickness=cv2.FILLED
    )  # Fill contours with white (255)

    # Invert the mask to highlight background regions
    iinverted_mask = cv2.bitwise_not(imask)

    # Color the background as orange
    ibackground_colored = cv2.cvtColor(iinverted_mask, cv2.COLOR_GRAY2BGR)
    ibackground_colored[
        np.where((ibackground_colored == [255, 255, 255]).all(axis=2))
    ] = [0, 165, 255]  # Orange in BGR format

    # Combine the original image with the colored background
    ifinal_image = cv2.addWeighted(ibackground_colored, 1, icontour, 1, 0)

    # Convert final image to RGB for visualization
    ifinal_image_rgb = cv2.cvtColor(ifinal_image, cv2.COLOR_BGR2RGB)

    for nn, (ifnn, icontoursn) in enumerate(icontoured_images.items()):
        # Sum of areas of all spots
        ispot_area = sum(cv2.contourArea(d) for d in icontoursn)
        # Calculate percentage
        ipercentage = (ispot_area / total_area) * 100
        iresult = f"Spots cover {ipercentage:.1f}% of area"

    axesi[1, im].imshow(ifinal_image_rgb)  # , cmap='gray'
    axesi[1, im].set_title(os.path.basename(ifnn))
    axesi[1, im].set_xlabel(iresult)  # Set the x-axis label

# ======================================

plt.tight_layout()  # Adjust subplot spacing
plt.show()